In [ ]:
import os
import json
import torch
import heapq
from tqdm import tqdm
from tqdm import trange
from datasets import load_dataset
import information_geometry as ig

base_path = "LLM_BASE_PATH" # Replace with the actual base path where data is stored
os.makedirs(base_path, exist_ok=True)

MODEL_NAME = "google/gemma-3-4b-pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

model, tokenizer, vocab_dict, vocab_list, G = ig.load_model_and_vocab(MODEL_NAME, device=DEVICE)

In [ ]:
#### Extract top 10k longest articles from C4 dataset (validation split) for English and French ####
def top_longest_articles_from_stream(dataset_name, lang_code, num_samples=10000, max_scan=1e7):
    ds = load_dataset(dataset_name, lang_code, split="validation", streaming=True)
    top_articles = []
    processed_count = 0

    print(f"\nScanning {dataset_name} ({lang_code}) validation split...")

    for example in tqdm(ds, desc=f"Scanning {lang_code}"):
        text = example['text']
        text_length = len(text)
        processed_count += 1
        
        if len(top_articles) < num_samples:
            heapq.heappush(top_articles, (text_length, text))
        elif text_length > top_articles[0][0]:
            heapq.heapreplace(top_articles, (text_length, text))
        
        if processed_count >= max_scan:
            break
            
    top_articles.sort(key=lambda x: x[0], reverse=True)
    return [article[1] for article in top_articles]

en_texts = top_longest_articles_from_stream("allenai/c4", "en")
fr_texts = top_longest_articles_from_stream("allenai/c4", "fr")

with open(os.path.join(base_path, 'texts_10k_en.json'), 'w', encoding='utf-8') as f:
    json.dump(en_texts, f, ensure_ascii=False, indent=4)

with open(os.path.join(base_path, 'texts_10k_fr.json'), 'w', encoding='utf-8') as f:
    json.dump(fr_texts, f, ensure_ascii=False, indent=4)

In [ ]:
#### Save texts in shards for embedding computation ###
def save_text_shards(texts, num_shards, lang, base_path):
    shard_size = len(texts) // num_shards
    os.makedirs(os.path.join(base_path, 'embeddings'), exist_ok=True)
    
    for shard_idx in trange(num_shards):
        start_idx = shard_idx * shard_size
        end_idx = len(texts) if shard_idx == num_shards - 1 else (shard_idx + 1) * shard_size
        
        shard_texts = texts[start_idx:end_idx]
        with open(os.path.join(base_path, f'embeddings/{lang}_texts_shard_{shard_idx}.json'), 'w') as f:
            json.dump(shard_texts, f)


save_text_shards(en_texts, num_shards=5, lang='en', base_path=base_path)
save_text_shards(fr_texts, num_shards=5, lang='fr', base_path=base_path)

### Then, run llm_embedding.py to compute and save embeddings for each shard and language. ###

In [ ]:
all_en_embeddings = []
for shard_ind in range(5):
    shard_path = os.path.join(base_path, f'embeddings/en_embeddings_shard_{shard_ind}.pt')
    shard_data = torch.load(shard_path)
    all_en_embeddings.append(shard_data)

In [ ]:
all_fr_embeddings = []
for shard_ind in range(5):
    shard_path = os.path.join(base_path, f'embeddings/fr_embeddings_shard_{shard_ind}.pt')
    shard_data = torch.load(shard_path)
    all_fr_embeddings.append(shard_data)

In [ ]:
def filter_indices_batch(embeddings, vocab_list, keys):
    new_indices = []
    for i in range(len(embeddings['topk_ids'])):
        topk_indices = embeddings['topk_ids'][i][:3]
        topk_values = torch.tensor(embeddings['topk_probs'][i][:3])
        if all(vocab_list[idx] in keys for idx in topk_indices):
            if topk_values.sum() > 0.7:
                new_indices.append(i)
    return new_indices

In [ ]:
concept_names = ["verb_en_fr", "verb_ing", "verb_past", "verb_third"]

#### Prepare datasets for each concept ####
for concept_name in concept_names:
    torch.random.manual_seed(100)
    print(f"\nProcessing concept: {concept_name}")

    concept_path = os.path.join(base_path, concept_name)
    os.makedirs(concept_path, exist_ok=True)

    with open(os.path.join(f"data/mapping_{concept_name}.json"), "r") as f:
        mapping = json.load(f)
    mapping = ig.get_clean_mapping(mapping, vocab_dict)
    
    shard_idx0 = []
    shard_idx1 = []
    embeddings0 = []
    embeddings1 = []
    text_idx0 = []
    text_idx1 = []
    token_pos0 = []
    token_pos1 = []
    for i in range(5):
        indices0 = filter_indices_batch(all_en_embeddings[i], vocab_list, list(mapping.keys()))
        embeddings0.append(all_en_embeddings[i]['embeddings'][indices0])
        text_idx0.append(all_en_embeddings[i]['text_idx'][indices0])
        token_pos0.append(all_en_embeddings[i]['token_pos'][indices0])
        shard_idx0.append(i * torch.ones(len(indices0), dtype=torch.long))

        if concept_name == "verb_en_fr":
            indices1 = filter_indices_batch(all_fr_embeddings[i], vocab_list, list(mapping.values()))
            embeddings1.append(all_fr_embeddings[i]['embeddings'][indices1])
            text_idx1.append(all_fr_embeddings[i]['text_idx'][indices1])
            token_pos1.append(all_fr_embeddings[i]['token_pos'][indices1])
            shard_idx1.append(i * torch.ones(len(indices1), dtype=torch.long))
        else:
            indices1 = filter_indices_batch(all_en_embeddings[i], vocab_list, list(mapping.values()))
            embeddings1.append(all_en_embeddings[i]['embeddings'][indices1])
            text_idx1.append(all_en_embeddings[i]['text_idx'][indices1])
            token_pos1.append(all_en_embeddings[i]['token_pos'][indices1])
            shard_idx1.append(i * torch.ones(len(indices1), dtype=torch.long))
        print(f"{len(indices0)} base indices, {len(indices1)} target indices in shard {i}.")

    embeddings0 = torch.cat(embeddings0, dim=0)
    embeddings1 = torch.cat(embeddings1, dim=0)
    text_idx0 = torch.cat(text_idx0, dim=0)
    text_idx1 = torch.cat(text_idx1, dim=0)
    token_pos0 = torch.cat(token_pos0, dim=0)
    token_pos1 = torch.cat(token_pos1, dim=0)
    shard_idx0 = torch.cat(shard_idx0, dim=0)
    shard_idx1 = torch.cat(shard_idx1, dim=0)

    num_train = 400
    num_test = 100

    if len(embeddings0) < num_train + num_test or len(embeddings1) < num_train + num_test:
        print(f"Not enough embeddings for concept {concept_name}. Skipping.")
        continue

    base_perm = torch.randperm(len(embeddings0))[:num_train + num_test]
    train_embeddings0 = embeddings0[base_perm[:num_train]]
    test_embeddings0 = embeddings0[base_perm[num_train:]]
    

    target_perm = torch.randperm(len(embeddings1))[:num_train + num_test]
    train_embeddings1 = embeddings1[target_perm[:num_train]]
    test_embeddings1 = embeddings1[target_perm[num_train:]]

    text_indices = {
        "train_idx0": (shard_idx0[base_perm[:num_train]],
                       text_idx0[base_perm[:num_train]],
                       token_pos0[base_perm[:num_train]]),
        "test_idx0": (shard_idx0[base_perm[num_train:]],
                      text_idx0[base_perm[num_train:]],
                      token_pos0[base_perm[num_train:]]),
        "train_idx1": (shard_idx1[target_perm[:num_train]],
                       text_idx1[target_perm[:num_train]],
                       token_pos1[target_perm[:num_train]]),
        "test_idx1": (shard_idx1[target_perm[num_train:]],
                      text_idx1[target_perm[num_train:]],
                      token_pos1[target_perm[num_train:]])
    }

    print(len(train_embeddings0), len(test_embeddings0), len(train_embeddings1), len(test_embeddings1))

    torch.save(train_embeddings0, os.path.join(concept_path, "train_embeddings0.pt"))
    torch.save(test_embeddings0 , os.path.join(concept_path, "test_embeddings0.pt"))
    torch.save(train_embeddings1, os.path.join(concept_path, "train_embeddings1.pt"))
    torch.save(test_embeddings1, os.path.join(concept_path, "test_embeddings1.pt"))
    torch.save(text_indices, os.path.join(concept_path, "text_indices.pt"))